In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

COLS = ["datetime", "open", "high", "low", "close", "volume"]

In [2]:
def load_ai_agent(path):
    ai = pd.read_csv(path, header=None)
    ai.columns = ["excel_date", "hour", "minute", "ref_price", "inventory"]

    # Excel serial date -> pandas datetime
    ai["date"] = pd.to_datetime("1899-12-30") + pd.to_timedelta(ai["excel_date"], unit="D")

    ai["datetime"] = (
        ai["date"]
        + pd.to_timedelta(ai["hour"], unit="h")
        + pd.to_timedelta(ai["minute"], unit="m")
    )

    ai = ai.sort_values("datetime").reset_index(drop=True)

    return ai

ai = load_ai_agent("AIAgent_EuroStoxx.csv")

In [3]:
# Functions for Loading and Cleaning the Data


# Reads one contract csv, fixes datatypes, sorts by time, and adds contract/date columns.
def load_contract(path, name, cols=COLS):
    df = pd.read_csv(path, header=None, names=cols)

    df["datetime"] = pd.to_datetime(
        df["datetime"],
        format="%Y.%m.%d.%H:%M:%S",
        errors="coerce"
    )

    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = (
        df.sort_values("datetime")
          .drop_duplicates(subset="datetime")
          .reset_index(drop=True)
    )

    df["contract"] = name
    df["date"] = df["datetime"].dt.normalize()

    return df

# Reads a group of contract files into a dictionary so the same code works for any asset.
def load_contracts(file_map):
    return {
        name: load_contract(path, name)
        for name, path in file_map.items()
    }

# Drops missing or clearly invalid OHLCV rows.
def remove_bad_rows(df):
    df = df.dropna(subset=["datetime", "open", "high", "low", "close", "volume"]).copy()

    good = (
        (df["high"] >= df["low"]) &
        (df["high"] >= df["open"]) &
        (df["high"] >= df["close"]) &
        (df["low"] <= df["open"]) &
        (df["low"] <= df["close"]) &
        (df["volume"] >= 0)
    )

    return df.loc[good].copy()

# Builds a daily summary with row count, total volume, average volume, and open/close.
def make_daily_summary(df):
    return (
        df.groupby("date", as_index=False)
          .agg(
              n_rows=("datetime", "size"),
              total_volume=("volume", "sum"),
              avg_volume=("volume", "mean"),
              open_price=("open", "first"),
              close_price=("close", "last")
          )
          .sort_values("date")
          .reset_index(drop=True)
    )

# Keeps only active trading days based on minimum row count and daily volume.
def filter_active_days(df, min_rows=100, min_daily_volume=1000):
    daily = make_daily_summary(df)

    keep_mask = (
        (daily["n_rows"] >= min_rows) &
        (daily["total_volume"] >= min_daily_volume)
    )

    keep_dates = set(daily.loc[keep_mask, "date"])

    df_clean = df[df["date"].isin(keep_dates)].copy()
    daily_clean = make_daily_summary(df_clean)

    return df_clean, daily, daily_clean

# Runs the main cleaning steps for one contract and returns both raw and cleaned daily summaries.
def clean_contract(df, min_rows=300, min_daily_volume=2000):
    df = remove_bad_rows(df)
    df_clean, daily, daily_clean = filter_active_days(
        df,
        min_rows=min_rows,
        min_daily_volume=min_daily_volume
    )
    return df_clean, daily, daily_clean

# Runs the same cleaning process for every contract in the dictionary.
def clean_contracts(contract_dict, min_rows=300, min_daily_volume=2000):
    cleaned = {}
    daily_raw = {}
    daily_clean = {}

    for name, df in contract_dict.items():
        df_clean, daily, daily_kept = clean_contract(
            df,
            min_rows=min_rows,
            min_daily_volume=min_daily_volume
        )
        cleaned[name] = df_clean
        daily_raw[name] = daily
        daily_clean[name] = daily_kept

    return cleaned, daily_raw, daily_clean

# Plots daily volume for two contracts so we can compare which one is more active.
def plot_daily_volume(daily_df, title, kept_dates=None, line_color="C0", point_color=None):
    if point_color is None:
        point_color = line_color

    plt.figure(figsize=(14, 4))
    plt.plot(
        daily_df["date"],
        daily_df["total_volume"],
        marker="o",
        markersize=3,
        color=line_color
    )

    if kept_dates is not None:
        kept = daily_df[daily_df["date"].isin(kept_dates)]
        plt.scatter(
            kept["date"],
            kept["total_volume"],
            s=25,
            color=point_color
        )

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Daily Volume")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Plots daily volume for two contracts so we can compare which one is more active.
def plot_two_contracts(daily_a, label_a, daily_b, label_b, title):
    plt.figure(figsize=(14, 4))
    plt.plot(daily_a["date"], daily_a["total_volume"], marker="o", markersize=3, label=label_a)
    plt.plot(daily_b["date"], daily_b["total_volume"], marker="o", markersize=3, label=label_b)
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Daily Volume")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



# Chooses the dominant contract each day based on total daily volume.
def choose_daily_dominant(daily_dict):
    all_daily = []

    for name, daily in daily_dict.items():
        temp = daily.copy()
        temp["contract"] = name
        all_daily.append(temp)

    all_daily = pd.concat(all_daily, ignore_index=True)

    dominant = (
        all_daily.sort_values(["date", "total_volume", "contract"], ascending=[True, False, True])
                 .drop_duplicates(subset="date")
                 .reset_index(drop=True)
    )
    return dominant


# Uses the dominant daily contract choice to stitch the minute data into one series.
def stitch_contracts_by_volume(contract_dict, daily_dict):
    dominant_daily = choose_daily_dominant(daily_dict)

    keep_map = dict(zip(dominant_daily["date"], dominant_daily["contract"]))

    stitched_parts = []
    for name, df in contract_dict.items():
        temp = df.copy()
        temp["dominant_contract"] = temp["date"].map(keep_map)
        temp = temp[temp["contract"] == temp["dominant_contract"]].copy()
        stitched_parts.append(temp)

    stitched = (
        pd.concat(stitched_parts, ignore_index=True)
          .sort_values("datetime")
          .drop_duplicates(subset="datetime")
          .reset_index(drop=True)
    )

    return stitched, dominant_daily


# Makes it easier to see when the dominant contract switches over time.
def plot_dominant_switch(dominant_daily, title="Dominant Contract by Daily Volume"):
    plt.figure(figsize=(14, 4))
    plt.plot(dominant_daily["date"], dominant_daily["total_volume"], marker="o", markersize=3)
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Winning Daily Volume")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [4]:
files = {
    "VGH22": "VGH22.csv",
    "VGM22": "VGM22.csv"
}

contracts = load_contracts(files)

cleaned_contracts, daily_raw, daily_clean = clean_contracts(
    contracts,
    min_rows=300,
    min_daily_volume=2000
)

stitched_vg, dominant_vg = stitch_contracts_by_volume(
    cleaned_contracts,
    daily_clean
)

stitched_vg.head()

,datetime,open,high,low,close,volume,contract,date,dominant_contract
0,2021-11-26 00:04:00,4179.0,4179.0,4179.0,4179.0,1,VGH22,2021-11-26,VGH22
1,2021-11-26 00:29:00,4175.0,4175.0,4175.0,4175.0,5,VGH22,2021-11-26,VGH22
2,2021-11-26 00:30:00,4175.0,4175.0,4175.0,4175.0,2,VGH22,2021-11-26,VGH22
3,2021-11-26 00:35:00,4179.0,4179.0,4179.0,4179.0,3,VGH22,2021-11-26,VGH22
4,2021-11-26 00:40:00,4181.5,4181.5,4181.5,4181.5,2,VGH22,2021-11-26,VGH22


In [4]:
# stitched_vg.to_csv("VG22_clean.csv", index=False)

stitched_vg.to_clipboard()

In [5]:
# Time windows / holding periods
# Tau / Ticks

# current: tau = 5, tick = 1
def add_tau_features(df, tau=5, tick_size=1.0):
    df = df.copy().sort_values("datetime").reset_index(drop=True)

    # future high/low over [t, t+tau)
    df[f"future_high_{tau}"] = (
        df["high"].rolling(window=tau, min_periods=tau).max().shift(-tau + 1)
    )
    df[f"future_low_{tau}"] = (
        df["low"].rolling(window=tau, min_periods=tau).min().shift(-tau + 1)
    )

    # project quantities
    df[f"range_up_{tau}"] = df[f"future_high_{tau}"] - df["open"]
    df[f"range_dn_{tau}"] = df["open"] - df[f"future_low_{tau}"]
    df[f"range_{tau}"] = df[f"future_high_{tau}"] - df[f"future_low_{tau}"]

    # convert to ticks
    df[f"range_up_{tau}_ticks"] = (df[f"range_up_{tau}"] / tick_size).round(6)
    df[f"range_dn_{tau}_ticks"] = (df[f"range_dn_{tau}"] / tick_size).round(6)
    df[f"range_{tau}_ticks"] = (df[f"range_{tau}"] / tick_size).round(6)

    return df


def add_state_features(df, vol_lookback=30, volu_lookback=30, trend_lookback=5):
    df = df.copy().sort_values("datetime").reset_index(drop=True)

    # 1-min return
    df["ret_1m"] = df["close"].pct_change()

    # rolling volatility and rolling average volume
    df["roll_vol"] = df["ret_1m"].rolling(vol_lookback).std()
    df["roll_avg_volume"] = df["volume"].rolling(volu_lookback).mean()

    # short trend
    df["trend_ret"] = df["close"].pct_change(trend_lookback)

    # simple binary states
    vol_med = df["roll_vol"].median()
    volu_med = df["roll_avg_volume"].median()

    df["vol_state"] = df["roll_vol"].apply(
        lambda x: "high_vol" if pd.notna(x) and x > vol_med else "low_vol"
    )
    df["volume_state"] = df["roll_avg_volume"].apply(
        lambda x: "high_volume" if pd.notna(x) and x > volu_med else "low_volume"
    )

    def trend_label(x):
        if pd.isna(x):
            return "flat"
        if x > 0:
            return "up"
        if x < 0:
            return "down"
        return "flat"

    df["trend_state"] = df["trend_ret"].apply(trend_label)

    return df

In [6]:
# Notes:
# The new df, vg_exec s

TAU = 5
TICK_SIZE = 1.0   # replace with correct EuroStoxx tick size

vg_exec = add_tau_features(stitched_vg, tau=TAU, tick_size=TICK_SIZE)
vg_exec = add_state_features(vg_exec)

vg_exec.head()
#vg_exec.to_clipboard()
#vg_exec.head(5000).to_clipboard(index=False)


,datetime,open,high,low,close,volume,contract,date,dominant_contract,future_high_5,...,range_up_5_ticks,range_dn_5_ticks,range_5_ticks,ret_1m,roll_vol,roll_avg_volume,trend_ret,vol_state,volume_state,trend_state
0,2021-11-26 00:04:00,4179.0,4179.0,4179.0,4179.0,1,VGH22,2021-11-26,VGH22,4181.5,...,2.5,4.0,6.5,NaN,NaN,NaN,NaN,low_vol,low_volume,flat
1,2021-11-26 00:29:00,4175.0,4175.0,4175.0,4175.0,5,VGH22,2021-11-26,VGH22,4181.5,...,6.5,0.0,6.5,-0.000957,NaN,NaN,NaN,low_vol,low_volume,flat
2,2021-11-26 00:30:00,4175.0,4175.0,4175.0,4175.0,2,VGH22,2021-11-26,VGH22,4185.5,...,10.5,0.0,10.5,0.000000,NaN,NaN,NaN,low_vol,low_volume,flat
3,2021-11-26 00:35:00,4179.0,4179.0,4179.0,4179.0,3,VGH22,2021-11-26,VGH22,4188.0,...,9.0,0.0,9.0,0.000958,NaN,NaN,NaN,low_vol,low_volume,flat
4,2021-11-26 00:40:00,4181.5,4181.5,4181.5,4181.5,2,VGH22,2021-11-26,VGH22,4189.0,...,7.5,0.0,7.5,0.000598,NaN,NaN,NaN,low_vol,low_volume,flat


In [7]:
def rule_cross_buy(row):
    return 0

def rule_cross_sell(row):
    return 0


def rule_one_tick_buy(row):
    return 1

def rule_one_tick_sell(row):
    return 1


def rule_one_point_five_buy(row):
    return 1.75

def rule_one_point_five_sell(row):
    return 1.75


def rule_two_tick_buy(row):
    return 2

def rule_two_tick_sell(row):
    return 2


def rule_vol_buy(row):
    if row["vol_state"] == "high_vol":
        return 0
    return 2

def rule_vol_sell(row):
    if row["vol_state"] == "high_vol":
        return 0
    return 2


def rule_state_buy(row):
    if row["vol_state"] == "low_vol" and row["volume_state"] == "high_volume":
        depth = 2
    elif row["vol_state"] == "high_vol" and row["volume_state"] == "low_volume":
        depth = 0
    else:
        depth = 1

    if row["trend_state"] == "up":
        depth -= 1
    elif row["trend_state"] == "down":
        depth += 1

    return max(0, min(depth, 3))


def rule_state_sell(row):
    if row["vol_state"] == "low_vol" and row["volume_state"] == "high_volume":
        depth = 2
    elif row["vol_state"] == "high_vol" and row["volume_state"] == "low_volume":
        depth = 0
    else:
        depth = 1

    if row["trend_state"] == "down":
        depth -= 1
    elif row["trend_state"] == "up":
        depth += 1

    return max(0, min(depth, 3))

In [8]:
# Test Rules

def backtest_rule_pair(df, buy_rule, sell_rule, tau=5, tick_size=1.0, rule_name="rule"):
    out = df.copy()

    # choose depths
    out["buy_depth"] = out.apply(buy_rule, axis=1)
    out["sell_depth"] = out.apply(sell_rule, axis=1)

    # build limit prices
    out["buy_limit_price"] = out["open"] - out["buy_depth"] * tick_size
    out["sell_limit_price"] = out["open"] + out["sell_depth"] * tick_size

    # fill logic
    out["buy_filled"] = out[f"range_dn_{tau}_ticks"] >= out["buy_depth"]
    out["sell_filled"] = out[f"range_up_{tau}_ticks"] >= out["sell_depth"]

    # price improvement if filled
    out["buy_improvement"] = np.where(out["buy_filled"], out["buy_depth"] * tick_size, np.nan)
    out["sell_improvement"] = np.where(out["sell_filled"], out["sell_depth"] * tick_size, np.nan)

    # summary
    summary = {
        "rule": rule_name,
        "buy_fill_rate": out["buy_filled"].mean(),
        "sell_fill_rate": out["sell_filled"].mean(),
        "buy_avg_improvement": out["buy_improvement"].mean(),
        "sell_avg_improvement": out["sell_improvement"].mean(),
        "avg_buy_depth": out["buy_depth"].mean(),
        "avg_sell_depth": out["sell_depth"].mean()
    }

    return out, summary



# Run the rules
cross_df, cross_summary = backtest_rule_pair(
    vg_exec,
    rule_cross_buy,
    rule_cross_sell,
    tau=TAU,
    tick_size=TICK_SIZE,
    rule_name="cross"
)

one_df, one_summary = backtest_rule_pair(
    vg_exec,
    rule_one_tick_buy,
    rule_one_tick_sell,
    tau=TAU,
    tick_size=TICK_SIZE,
    rule_name="one_tick"
)

two_df, two_summary = backtest_rule_pair(
    vg_exec,
    rule_two_tick_buy,
    rule_two_tick_sell,
    tau=TAU,
    tick_size=TICK_SIZE,
    rule_name="two_tick"
)

vol_df, vol_summary = backtest_rule_pair(
    vg_exec,
    rule_vol_buy,
    rule_vol_sell,
    tau=TAU,
    tick_size=TICK_SIZE,
    rule_name="vol_rule"
)

state_df, state_summary = backtest_rule_pair(
    vg_exec,
    rule_state_buy,
    rule_state_sell,
    tau=TAU,
    tick_size=TICK_SIZE,
    rule_name="state_rule"
)

one5_df, one5_summary = backtest_rule_pair(
    vg_exec,
    rule_one_point_five_buy,
    rule_one_point_five_sell,
    tau=TAU,
    tick_size=TICK_SIZE,
    rule_name="one_point_five_tick"
)

rule_results = pd.DataFrame([
    cross_summary,
    one_summary,
    one5_summary,
    two_summary,
    vol_summary,
    state_summary
])

rule_results

rule_results = pd.DataFrame([
    cross_summary,
    one_summary,
    one5_summary,
    two_summary,
    vol_summary,
    state_summary
])

rule_results

,rule,buy_fill_rate,sell_fill_rate,buy_avg_improvement,sell_avg_improvement,avg_buy_depth,avg_sell_depth
0,cross,0.999974,0.999974,0.000000,0.000000,0.000000,0.000000
1,one_tick,0.768778,0.772538,1.000000,1.000000,1.000000,1.000000
2,one_point_five_tick,0.541991,0.545259,1.750000,1.750000,1.750000,1.750000
3,two_tick,0.541991,0.545259,2.000000,2.000000,2.000000,2.000000
4,vol_rule,0.702723,0.706515,0.577327,0.584963,1.000203,1.000203
5,state_rule,0.789153,0.783193,0.823237,0.822127,1.054614,1.065008


In [9]:
rule_results["buy_miss_rate"] = 1 - rule_results["buy_fill_rate"]
rule_results["sell_miss_rate"] = 1 - rule_results["sell_fill_rate"]

rule_results = rule_results[[
    "rule",
    "buy_fill_rate", "sell_fill_rate",
    "buy_miss_rate", "sell_miss_rate",
    "buy_avg_improvement", "sell_avg_improvement",
    "avg_buy_depth", "avg_sell_depth"
]]

rule_results


def summarize_by_state(df, rule_name):
    out = df.groupby(["vol_state", "volume_state", "trend_state"])[
        ["buy_filled", "sell_filled", "buy_improvement", "sell_improvement"]
    ].mean().reset_index()

    out["rule"] = rule_name
    out["buy_miss_rate"] = 1 - out["buy_filled"]
    out["sell_miss_rate"] = 1 - out["sell_filled"]

    return out


cross_state_summary = summarize_by_state(cross_df, "cross")
one_state_summary   = summarize_by_state(one_df, "one_tick")
vol_state_summary   = summarize_by_state(vol_df, "vol_rule")
state_state_summary = summarize_by_state(state_df, "state_rule")

all_state_results = pd.concat([
    cross_state_summary,
    one_state_summary,
    vol_state_summary,
    state_state_summary
], ignore_index=True)

all_state_results.to_clipboard()

In [10]:
rule_results["buy_score"] = rule_results["buy_fill_rate"] + 0.25 * rule_results["buy_avg_improvement"]
rule_results["sell_score"] = rule_results["sell_fill_rate"] + 0.25 * rule_results["sell_avg_improvement"]
rule_results["total_score"] = rule_results["buy_score"] + rule_results["sell_score"]

rule_results.sort_values("total_score", ascending=False)

C:\Users\natej\AppData\Local\Temp\ipykernel_19208\3712450248.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rule_results["buy_score"] = rule_results["buy_fill_rate"] + 0.25 * rule_results["buy_avg_improvement"]
C:\Users\natej\AppData\Local\Temp\ipykernel_19208\3712450248.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rule_results["sell_score"] = rule_results["sell_fill_rate"] + 0.25 * rule_results["sell_avg_improvement"]
C:\Users\natej\AppData\Local\Temp\ipykernel_19208\3712450248.py:3: SettingWit

,rule,buy_fill_rate,sell_fill_rate,buy_miss_rate,sell_miss_rate,buy_avg_improvement,sell_avg_improvement,avg_buy_depth,avg_sell_depth,buy_score,sell_score,total_score
3,two_tick,0.541991,0.545259,0.458009,0.454741,2.000000,2.000000,2.000000,2.000000,1.041991,1.045259,2.087249
1,one_tick,0.768778,0.772538,0.231222,0.227462,1.000000,1.000000,1.000000,1.000000,1.018778,1.022538,2.041316
0,cross,0.999974,0.999974,0.000026,0.000026,0.000000,0.000000,0.000000,0.000000,0.999974,0.999974,1.999948
5,state_rule,0.789153,0.783193,0.210847,0.216807,0.823237,0.822127,1.054614,1.065008,0.994962,0.988725,1.983687
2,one_point_five_tick,0.541991,0.545259,0.458009,0.454741,1.750000,1.750000,1.750000,1.750000,0.979491,0.982759,1.962249
4,vol_rule,0.702723,0.706515,0.297277,0.293485,0.577327,0.584963,1.000203,1.000203,0.847054,0.852755,1.699809


In [15]:
one_state_summary = summarize_by_state(one_df, "cross")
state_state_summary = summarize_by_state(state_df, "state_rule")


compare_states = one_state_summary.merge(
    state_state_summary,
    on=["vol_state", "volume_state", "trend_state"],
    suffixes=("_one", "_state")
)

compare_states["buy_fill_diff"] = compare_states["buy_filled_state"] - compare_states["buy_filled_one"]
compare_states["sell_fill_diff"] = compare_states["sell_filled_state"] - compare_states["sell_filled_one"]

compare_states["buy_improve_diff"] = compare_states["buy_improvement_state"] - compare_states["buy_improvement_one"]
compare_states["sell_improve_diff"] = compare_states["sell_improvement_state"] - compare_states["sell_improvement_one"]

compare_states.to_clipboard()